# 🧠💥🧠 Notebook 1 — Split Brain: Two Leaders at Once

## What you'll learn

- What a **split-brain** is in a distributed system.
- Why a *single* leader isn't as simple as "whoever grabs the lock first".
- How a slow **Garbage Collection (GC) pause** or **network partition** can create two leaders who both *believe* they're in charge.
- Why this silently corrupts data in the real world.

> We're going to **reproduce the bug first** (the "BAD" version). The next notebook shows the fix.

## Analogy

Imagine a bank has one manager on duty at a time, and the manager holds the vault key. One morning the manager gets stuck in a traffic jam (the "GC pause"). The bank can't reach them, assumes they're sick, and promotes a new manager who gets a second key made.

An hour later the first manager arrives — still holding the original key, still believing *they* are the manager. Now **two people with keys** are both making decisions about the same vault. That's split brain.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/split-brain-and-fencing
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: just use a lock, what could go wrong?

A lock service gives out a lease (a lock with an expiry). The first node to grab it is "the leader". Simple.

Below, we model:

- A tiny **`Storage`** — the shared resource every leader writes to (think: the config database, the payment ledger, the Kafka controller znode).
- Two candidate leaders, **A** and **B**.
- A realistic failure sequence: A gets the lease → A pauses → the cluster thinks A is dead → B takes over → A wakes up and **keeps writing as if nothing happened**.

Storage has **no idea** which writer is the "real" leader. It just appends whatever arrives.


In [1]:
from dataclasses import dataclass, field
from typing import List

# The shared resource. Could be a config file, a database row, a Kafka topic, etc.
# It trusts every writer. That's the whole point of the BAD version.
@dataclass
class Storage:
    log: List[str] = field(default_factory=list)

    def write(self, who: str, value: str) -> None:
        self.log.append(f"{who:>2} -> {value}")
        print(f"  storage accepted write from {who}: {value!r}")


store = Storage()
print("step 1) A acquires the lease and starts leading")
leader = "A"
store.write(leader, "config v1")


step 1) A acquires the lease and starts leading
  storage accepted write from A: 'config v1'


In [2]:
print("step 2) A enters a long GC pause (or a network partition isolates it)")
print("        ... from the cluster's point of view, A is dead ...")
print()
print("step 3) the cluster elects B as the new leader")
leader = "B"
store.write(leader, "config v2")


step 2) A enters a long GC pause (or a network partition isolates it)
        ... from the cluster's point of view, A is dead ...

step 3) the cluster elects B as the new leader
  storage accepted write from B: 'config v2'


In [3]:
print("step 4) A wakes up. Its local clock says the lease is still valid.")
print("        A has no way to know the cluster moved on without it.")
print("        A writes again, because A still thinks it is the leader.")
store.write("A", "config v1.1 (STALE but A does not know)")

print()
print("step 5) B, the real leader, writes again")
store.write("B", "config v3")

print()
print("final storage log:")
for entry in store.log:
    print(" ", entry)


step 4) A wakes up. Its local clock says the lease is still valid.
        A has no way to know the cluster moved on without it.
        A writes again, because A still thinks it is the leader.
  storage accepted write from A: 'config v1.1 (STALE but A does not know)'

step 5) B, the real leader, writes again
  storage accepted write from B: 'config v3'

final storage log:
   A -> config v1
   B -> config v2
   A -> config v1.1 (STALE but A does not know)
   B -> config v3


### What just happened?

The storage layer accepted writes from **both** A and B, in an order that makes no sense:

```
A -> config v1
B -> config v2
A -> config v1.1  ← stale leader, but storage can't tell
B -> config v3
```

Whoever reads "the current config" gets whichever write landed last — not necessarily from the real leader.


## 💸 Why this is not just academic — a concrete money example

Let's replay the same bug, but now the "config" is a **bank balance**. Two leaders both process a withdrawal request from the same customer.


In [4]:
@dataclass
class Account:
    owner: str
    balance: int  # in cents

    def withdraw(self, who: str, amount: int) -> None:
        # No fencing. Anyone who claims to be leader can move money.
        self.balance -= amount
        print(f"  {who} withdrew {amount:>4} | balance is now {self.balance}")


acct = Account(owner="alice", balance=10_000)  # $100.00

# A is leader; processes a $70 withdrawal.
print("A (leader) processes alice's $70 withdrawal")
acct.withdraw("A", 7_000)

# Network hiccup. Cluster promotes B. Customer retries because they got no response.
# B (legitimately) processes the retry — it looks like a brand new request to B.
print("B (new leader) processes the same retry as a fresh $70 withdrawal")
acct.withdraw("B", 7_000)

# A wakes up from GC pause; finishes what it was doing.
# It never heard a response either, so it also retries internally.
print("A (stale leader, still thinks it is leader) also retries the $70 withdrawal")
acct.withdraw("A", 7_000)

print()
print(f"final balance: {acct.balance} cents  (should have been 3000)")
print("Alice just got charged $210 for a $70 purchase. 🎉 split brain 🎉")


A (leader) processes alice's $70 withdrawal
  A withdrew 7000 | balance is now 3000
B (new leader) processes the same retry as a fresh $70 withdrawal
  B withdrew 7000 | balance is now -4000
A (stale leader, still thinks it is leader) also retries the $70 withdrawal
  A withdrew 7000 | balance is now -11000

final balance: -11000 cents  (should have been 3000)
Alice just got charged $210 for a $70 purchase. 🎉 split brain 🎉


### Takeaways

- **Timeouts alone are not enough.** A node can always be "slow, but not dead".
- **The old leader doesn't know it's old.** From its own point of view, nothing is wrong.
- **Storage needs its own safety check.** It must be able to reject writes from stale leaders, without trusting the caller's opinion about who is leader.

👉 Next notebook: **fencing tokens** — a monotonically increasing number that lets the storage layer reject stale writers.


## 🌍 Real-world systems that hit this

| System | Where split-brain can happen | Mitigation |
|---|---|---|
| **HDFS NameNode HA** | Active NameNode pauses; standby becomes active | ZooKeeper + fencing (STONITH of the old NN) |
| **Kafka controller** | Controller broker isolated, another elected | Controller **epoch** (a fencing token) |
| **Redis Sentinel / Redlock** | Master partitioned from majority of sentinels | Failover + fencing tokens at the client |
| **PostgreSQL with repmgr / Patroni** | Two primaries after a flaky network | STONITH (kill the old primary) + WAL fencing |
| **Kubernetes leader election** | A controller pod GC-pauses past its lease | `resourceVersion` acts like a fencing token |

Every single one of these reaches for the same pattern: **don't trust the caller's claim to be leader — attach a number that goes up, and let the resource reject stale numbers.** That's notebook 2.
